In [ ]:
%pip install -r ./requirements.txt fpdf2
dbutils.library.restartPython()

# This is a one time run so keeping install of fppdf2 seperate and also getting fastmcp from requirements.txt


In [ ]:
import importlib
import json
from pathlib import Path
import sys

sys.dont_write_bytecode = True

for mod_name in list(sys.modules.keys()):
    if mod_name == "fra_safety_mcp" or mod_name.startswith("fra_safety_mcp.") or mod_name == "run_eval":
        del sys.modules[mod_name]

sys.path_importer_cache.clear()
importlib.invalidate_caches()

REPO_ROOT = Path.cwd().resolve().parent
SRC_PATH = str(REPO_ROOT / "src")
EVAL_PATH = str(REPO_ROOT / "eval")

for p in (SRC_PATH, EVAL_PATH):
    if p not in sys.path:
        sys.path.insert(0, p)

from fra_safety_mcp.config import config
from fra_safety_mcp.db import execute_query
from run_eval import run_benchmark

# 1. Execute all 14 MCP tool benchmark cases, 6 resources, and 4 prompts
eval_summary = run_benchmark()

# 2. Query live Unity Catalog row counts across all 4 FRA tables
table_stats = {}
for label, tbl in [
    ("Form 54 - Rail Equipment Accidents/Incidents", config.form_54_table),
    ("Form 55a - Railroad Injury & Illness Summary", config.form_55a_table),
    ("Form 55 - Operational Exposure (Train-Miles & Hours)", config.form_55_table),
    ("Form 57 - Highway-Rail Grade Crossing Incidents", config.form_57_table),
]:
    row = execute_query(f"SELECT COUNT(*) AS cnt FROM {config.fq_table(tbl)}")[0]
    table_stats[label] = int(row["cnt"])

print("Live Unity Catalog Row Counts:", json.dumps(table_stats, indent=2))
print(
    f"Evaluation Pass Rate: {eval_summary['tool_cases_passed']}/{eval_summary['tool_cases_total']} "
    f"({eval_summary['tool_pass_rate_pct']}%)"
)


In [ ]:
import datetime
from pathlib import Path
from fpdf import FPDF


class HackathonSubmissionPDF(FPDF):
    def header(self):
        self.set_fill_color(15, 44, 89)  # DOT / FRA Navy
        self.rect(0, 0, 210, 18, "F")
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(255, 255, 255)
        self.set_xy(12, 5)
        self.cell(
            0,
            8,
            "GSA HACKATHON | DATASET ACCESS MCP SERVER SUBMISSION - mcp-fra-safety",
            align="L",
        )
        self.ln(16)

    def footer(self):
        self.set_y(-13)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(100, 110, 125)
        self.cell(
            0,
            8,
            f"U.S. DOT / Federal Railroad Administration (49 CFR Part 225) | Page {self.page_no()}/{{nb}}",
            align="C",
        )

    def section_title(self, title: str):
        self.ln(3)
        self.set_font("Helvetica", "B", 12)
        self.set_text_color(15, 44, 89)
        self.cell(0, 7, title, new_x="LMARGIN", new_y="NEXT")
        self.set_draw_color(30, 100, 180)
        self.set_line_width(0.5)
        self.line(12, self.get_y(), 198, self.get_y())
        self.ln(3)

    def body_text(self, text: str):
        self.set_font("Helvetica", "", 9.5)
        self.set_text_color(35, 40, 50)
        self.multi_cell(0, 5, text)
        self.ln(1.5)

    def bullet_item(self, lead: str, text: str):
        self.set_font("Helvetica", "B", 9.5)
        self.set_text_color(15, 44, 89)
        self.cell(5, 5, "-")
        self.write(5, f"{lead}: ")
        self.set_font("Helvetica", "", 9.5)
        self.set_text_color(35, 40, 50)
        self.write(5, text)
        self.ln(5.5)


pdf = HackathonSubmissionPDF(orientation="P", unit="mm", format="A4")
pdf.alias_nb_pages()
pdf.set_margins(12, 22, 12)
pdf.set_auto_page_break(auto=True, margin=16)

# --- PAGE 1: Executive Summary & Data Foundation ---
pdf.add_page()
pdf.set_font("Helvetica", "B", 17)
pdf.set_text_color(15, 44, 89)
pdf.multi_cell(0, 8, "mcp-fra-safety: 49 CFR Part 225 Rail Safety Dataset Access MCP Server")
pdf.ln(1)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(80, 90, 105)
generated_dt = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
pdf.cell(
    0,
    6,
    f"Track: Dataset Access Server | Author: Neeraj Koul (U.S. DOT / FRA) | Generated: {generated_dt}",
    new_x="LMARGIN",
    new_y="NEXT",
)
pdf.ln(3)

# Key Metrics Callout Box
pdf.set_fill_color(240, 245, 252)
pdf.set_draw_color(180, 200, 225)
pdf.rect(12, pdf.get_y(), 186, 20, "DF")
y_box = pdf.get_y() + 3
pdf.set_xy(15, y_box)
pdf.set_font("Helvetica", "B", 10)
pdf.set_text_color(15, 44, 89)
total_rows = sum(table_stats.values())
pdf.cell(
    0,
    6,
    f"Governed Records: {total_rows:,} rows across 4 FRA Forms | "
    f"MCP Primitives: 8 Tools, 6 Resources, 4 Prompts | "
    f"Benchmark Score: {eval_summary['tool_cases_passed']}/{eval_summary['tool_cases_total']} "
    f"({eval_summary['tool_pass_rate_pct']}%)",
    new_x="LMARGIN",
    new_y="NEXT",
)
pdf.set_x(15)
pdf.set_font("Helvetica", "", 9)
pdf.set_text_color(50, 60, 75)
pdf.cell(
    0,
    6,
    f"Runtime: Databricks Apps (FastMCP Streamable-HTTP at /mcp) | "
    f"Catalog: {config.catalog}.{config.schema} | Avg Query Latency: {eval_summary['avg_tool_latency_ms']} ms",
)
pdf.set_y(y_box + 20)

pdf.section_title("1. Executive Summary & Regulatory Problem Solved")
pdf.body_text(
    "Federal Railroad Administration (FRA) safety data under 49 CFR Part 225 spans four interdependent "
    "reporting forms (Forms 54, 55, 55a, and 57). When general-purpose LLMs query raw FRA CSVs or SQL "
    "tables directly, they routinely produce flawed safety metrics: double-counting multi-train collisions "
    "where both railroads file Form 54 reports, mixing highway-rail crossing crashes into Report 2.03 "
    "train accident totals, or computing invalid state-level train-mile rates where Form 55 operational "
    "denominators are reported at the railroad-system level rather than by state."
)
pdf.body_text(
    "mcp-fra-safety solves this by exposing a governed, domain-accurate Model Context Protocol (MCP) server "
    "on Databricks Apps backed by Unity Catalog Delta tables. Every tool enforces official FRA Open Data "
    "portal rules (portals dsuf-xcni, wjun-6kqb, 9hnw-nt86, and filter glossary dnz7-ag8y), enables cross-form "
    "incident walking across Forms 54, 57, and 55a, and pairs executable SQL analytics with embedded "
    "49 CFR Part 225 methodology resources and multi-step prompts."
)

pdf.section_title("2. Unity Catalog Data Foundation (users.neeraj_koul)")
pdf.set_font("Helvetica", "B", 9)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(146, 7, "  Unity Catalog FRA Dataset", border=1, fill=True)
pdf.cell(40, 7, "Ingested Rows  ", border=1, fill=True, align="R", new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 9)
pdf.set_text_color(35, 40, 50)
for idx, (lbl, cnt) in enumerate(table_stats.items()):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    pdf.cell(146, 6.5, f"  {lbl}", border=1, fill=fill)
    pdf.cell(40, 6.5, f"{cnt:,}  ", border=1, fill=fill, align="R", new_x="LMARGIN", new_y="NEXT")

pdf.section_title("3. Key 49 CFR Part 225 Parity & Governance Guarantees")
pdf.bullet_item(
    "Multi-Train Collision Deduplication",
    "Counts distinct incidentkey values where jointcd = '1' for macro totals while summing totaldamagecost "
    "across all involved railroad filings (jointcd 1, 2, and 3) to prevent double-counting.",
)
pdf.bullet_item(
    "Geographic Blank-Rate Rule (Report 1.12)",
    "Automatically sets Form 55 operational exposure and normalized rates (per 1M train-miles or per 200K "
    "employee hours) to None whenever state, county, or city filters are applied, matching FRA portal dsuf-xcni.",
)
pdf.bullet_item(
    "Calendar vs. Federal Fiscal Year",
    "Supports both Calendar Year (Jan-Dec) and Federal Fiscal Year (Oct 1 - Sep 30) across all four FRA forms.",
)
pdf.bullet_item(
    "Cross-Form Incident Walking & Governed SQL",
    "Walks linked events across Form 54 (incidentkey, gxid), Form 57 (gradecrossingid), and Form 55a "
    "(individual casualties), with read-only SQL validation and explicit row-truncation telemetry.",
)

# --- PAGE 2: MCP Architecture & Primitives Catalog ---
pdf.add_page()
pdf.section_title("4. MCP Server Architecture & Primitives (8 Tools, 6 Resources, 4 Prompts)")
pdf.body_text(
    "Built with FastMCP and deployed on Databricks Apps as a Streamable-HTTP ASGI service (/mcp), "
    "mcp-fra-safety authenticates to Databricks SQL Warehouse <DATABRICKS_WAREHOUSE_ID> via zero-secret OAuth M2M "
    "service principal credentials and executes parameterized SQL directly against Unity Catalog Delta tables."
)

tools_catalog = [
    ("get_report_1_12_rail_safety_overview", "Forms 54/55a/57/55", "Replicates FRA Report 1.12 overview & normalized rates."),
    ("get_train_accident_summary", "Form 54", "Replicates Reports 2.03, 3.10 & 3.18 (Table 1 macro & Table 2 carrier)."),
    ("audit_joint_track_cause_discrepancies", "Form 54", "Surfaces primary vs. secondary railroad cause & speed mismatches."),
    ("analyze_highway_rail_crossings", "Form 57", "Replicates Reports 1.11, 2.08, 5.14, 5.15 & USDOT crossing hotspots."),
    ("analyze_metro_corridor_safety", "Forms 54/55a/57", "Synthesizes multi-portal rail & crossing risk across metro corridors."),
    ("audit_delta_table_versions", "Unity Catalog", "Tracks Delta DESCRIBE HISTORY & VERSION AS OF retroactive updates."),
    ("walk_across_fra_forms", "Forms 54/57/55a", "Walks linked incidents across accidents, crossings & individual casualties."),
    ("run_fra_sql_query", "All 4 FRA Forms", "Executes read-only cross-form SQL JOINs with truncation alerts."),
]

pdf.set_font("Helvetica", "B", 8.5)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(68, 7, "  MCP Tool Name", border=1, fill=True)
pdf.cell(32, 7, "  Forms", border=1, fill=True)
pdf.cell(86, 7, "  Regulatory & Analytical Scope", border=1, fill=True, new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 8.2)
pdf.set_text_color(35, 40, 50)
for idx, (t_name, t_forms, t_desc) in enumerate(tools_catalog):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    pdf.cell(68, 6.5, f"  {t_name}", border=1, fill=fill)
    pdf.cell(32, 6.5, f"  {t_forms}", border=1, fill=fill)
    pdf.cell(86, 6.5, f"  {t_desc}", border=1, fill=fill, new_x="LMARGIN", new_y="NEXT")

pdf.ln(4)
pdf.bullet_item(
    "6 Methodology Resources (fra://...)",
    "fra://reference/preparing-reports-guide, fra://reports/1.12-methodology, "
    "fra://reports/train-accidents-methodology, fra://reports/form57-crossings-methodology, "
    "fra://reference/filter-glossary, and fra://reference/cause-code-categories.",
)
pdf.bullet_item(
    "4 Workflow Prompts",
    "metro_safety_audit_briefing, railroad_system_safety_scorecard, "
    "grade_crossing_hotspot_audit, and joint_track_discrepancy_review.",
)

# --- PAGE 3: Automated Evaluation Results ---
pdf.add_page()
pdf.section_title("5. Live Unity Catalog Evaluation Benchmark Results")
pdf.body_text(
    f"The automated benchmark suite (eval/run_eval.py) executed {eval_summary['tool_cases_total']} end-to-end "
    f"tool evaluations, {eval_summary['resources_total']} resource reads, and {eval_summary['prompts_total']} "
    f"prompt templates directly against Unity Catalog in {eval_summary['total_duration_s']}s with a "
    f"{eval_summary['tool_pass_rate_pct']}% pass rate."
)

pdf.set_font("Helvetica", "B", 8.5)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(62, 7, "  Benchmark Case ID", border=1, fill=True)
pdf.cell(78, 7, "  Parity Verification Category", border=1, fill=True)
pdf.cell(24, 7, "Latency  ", border=1, fill=True, align="R")
pdf.cell(22, 7, "Status", border=1, fill=True, align="C", new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 8)
pdf.set_text_color(35, 40, 50)
for idx, r in enumerate(eval_summary["case_results"]):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    status_lbl = "PASS" if r["passed"] else "FAIL"
    pdf.cell(62, 6, f"  {r['id']}", border=1, fill=fill)
    pdf.cell(78, 6, f"  {r['category']}", border=1, fill=fill)
    pdf.cell(24, 6, f"{r['latency_ms']} ms  ", border=1, fill=fill, align="R")
    pdf.cell(22, 6, status_lbl, border=1, fill=fill, align="C", new_x="LMARGIN", new_y="NEXT")

out_tmp = Path("/tmp/mcp-fra-safety_GSA_Hackathon_Submission.pdf")
out_ws = REPO_ROOT / "mcp-fra-safety_GSA_Hackathon_Submission.pdf"

pdf.output(str(out_tmp))
try:
    pdf.output(str(out_ws))
    print(f"Saved Submission PDF to Workspace: {out_ws}")
except OSError:
    print(f"Saved Submission PDF to /tmp: {out_tmp}")


In [ ]:
import ast
import json
from pathlib import Path

# 1. Locate repository root dynamically
cwd = Path.cwd().resolve()
if (cwd / "src" / "fra_safety_mcp").is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "src" / "fra_safety_mcp").is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Run this cell from a notebook inside mcp-fra-safety (current dir: {cwd}).")

PKG_DIR = REPO_ROOT / "src" / "fra_safety_mcp"
TOOLS_DIR = PKG_DIR / "tools"

# 2. Define file contents (using # comments inside strings to prevent quote collisions)
GITIGNORE_CONTENT = '''# Local environment & credentials
.env
.env.*
!.env.example

# Generated artifacts & scratch directories
*.pdf
eval/results.json
misc/

# Python cache & virtual environments
__pycache__/
*.py[cod]
.pytest_cache/
.venv/
'''

ENV_EXAMPLE_CONTENT = '''# Databricks Workspace & SQL Warehouse (leave blank inside Databricks Notebooks / Apps)
DATABRICKS_HOST=https://<your-workspace>.cloud.databricks.com
DATABRICKS_TOKEN=
DATABRICKS_WAREHOUSE_ID=<your-sql-warehouse-id>

# Unity Catalog Target
FRA_CATALOG=main
FRA_SCHEMA=fra_safety

# FRA Part 225 & STB Part 1246 Table Names
FRA_FORM_54_TABLE=form_54_accidents
FRA_FORM_55A_TABLE=form_55a_casualties
FRA_FORM_55_TABLE=form_55_operational
FRA_FORM_57_TABLE=form_57_crossings
FRA_STB_1246_TABLE=stb_1246_all_carriers

# Live App Endpoint (used only by eval/verify_live_endpoint.py)
MCP_APP_URL=https://<your-app-name>.aws.databricksapps.com
'''

ENV_LOCAL_CONTENT = '''# Local Databricks configuration (ignored by Git via .gitignore)
DATABRICKS_WAREHOUSE_ID=<DATABRICKS_WAREHOUSE_ID>
FRA_CATALOG=users
FRA_SCHEMA=neeraj_koul
FRA_FORM_54_TABLE=form_54_accidents
FRA_FORM_55A_TABLE=form_55a_casualties
FRA_FORM_55_TABLE=form_55_operational
FRA_FORM_57_TABLE=form_57_crossings
FRA_STB_1246_TABLE=stb_1246_all_carriers
MCP_APP_URL=https://<your-app>.aws.databricksapps.com
'''

CONFIG_PY_CONTENT = '''import os
from dataclasses import dataclass
from pathlib import Path


def _load_dotenv() -> None:
    # Loads key=value pairs from .env at the repo root without overriding existing env vars
    candidates = [
        Path(__file__).resolve().parents[2] / ".env",
        Path.cwd() / ".env",
    ]
    for env_path in candidates:
        if not env_path.is_file():
            continue
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[len("export "):].strip()
            key, val = line.split("=", 1)
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            if key and val:
                os.environ.setdefault(key, val)
        break


_load_dotenv()


@dataclass(frozen=True)
class ServerConfig:
    catalog: str = os.getenv("FRA_CATALOG", "main")
    schema: str = os.getenv("FRA_SCHEMA", "fra_safety")
    warehouse_id: str = os.getenv("DATABRICKS_WAREHOUSE_ID", "")
    app_url: str = os.getenv("MCP_APP_URL", "")
    form_54_table: str = os.getenv("FRA_FORM_54_TABLE", "form_54_accidents")
    form_55a_table: str = os.getenv("FRA_FORM_55A_TABLE", "form_55a_casualties")
    form_55_table: str = os.getenv("FRA_FORM_55_TABLE", "form_55_operational")
    form_57_table: str = os.getenv("FRA_FORM_57_TABLE", "form_57_crossings")
    stb_1246_table: str = os.getenv("FRA_STB_1246_TABLE", "stb_1246_all_carriers")

    def fq_table(self, table_name: str) -> str:
        return f"`{self.catalog}`.`{self.schema}`.`{table_name}`"


config = ServerConfig()
'''

STB_EMPLOYMENT_CODE = '''from typing import Any, Dict, Optional
from fastmcp import FastMCP

from ..config import config
from ..db import DatabaseQueryError, execute_query, get_table_columns, resolve_col
from ..queries import build_common_filters, build_form54_filters, build_time_predicate

_FRA_TO_STB_MARK: Dict[str, str] = {
    "CSX": "CSXT",
    "CSXT": "CSXT",
    "CN": "GTC",
    "GTW": "GTC",
    "IC": "GTC",
    "WC": "GTC",
    "GTC": "GTC",
    "ATK": "AMTK",
    "AMTK": "AMTK",
    "CPKC": "CPKC",
    "CP": "CPKC",
    "SOO": "SOO",
    "KCS": "KCS",
    "BNSF": "BNSF",
    "NS": "NS",
    "UP": "UP",
}

_STB_TO_FRA_PARENT: Dict[str, str] = {
    "CSXT": "CSX",
    "GTC": "CN",
    "AMTK": "ATK",
    "CPKC": "CPKC",
    "SOO": "CPKC",
    "KCS": "CPKC",
    "BNSF": "BNSF",
    "NS": "NS",
    "UP": "UP",
}


def register(mcp: FastMCP) -> None:
    @mcp.tool()
    def analyze_class1_employment_and_safety(
        start_year: int = 2020,
        end_year: int = 2025,
        railroad_code: Optional[str] = None,
        sub_measure_prefix: Optional[str] = None,
        state: Optional[str] = None,
    ) -> Dict[str, Any]:
        # Queries STB Part 1246 Class I monthly employment and cross-links FRA Part 225 safety rates
        params: Dict[str, Any] = {}
        raw_rr = railroad_code.strip().upper() if railroad_code else None
        stb_mark = _FRA_TO_STB_MARK.get(raw_rr, raw_rr) if raw_rr else None
        fra_parent = _STB_TO_FRA_PARENT.get(stb_mark, stb_mark) if stb_mark else None

        try:
            cstb = get_table_columns(config.stb_1246_table)
            aar_col = resolve_col(
                cstb, "e", ("railroad_reporting_aar_mark", "aar_mark", "railroad_code", "reporting_railroad_code")
            )
            end_dt_col = resolve_col(
                cstb, "e", ("report_period_end_date", "report_period_start_date", "period_end_date", "date")
            )
            meas_col = resolve_col(cstb, "e", ("measure_name", "measure"), fallback_sql="'Employees mid-month'")
            sub_col = resolve_col(cstb, "e", ("sub_measure_name", "sub_measure"), fallback_sql="'ALL'")
            val_col = resolve_col(cstb, "e", ("fact_value", "value", "employee_count"))

            dt_expr = f"COALESCE(TRY_CAST({end_dt_col} AS DATE), TO_DATE(CAST({end_dt_col} AS STRING), 'MM/dd/yyyy'))"
            yr_expr = f"YEAR({dt_expr})"
            mo_expr = f"MONTH({dt_expr})"
            num_val_expr = f"TRY_CAST(REPLACE(CAST({val_col} AS STRING), ',', '') AS DOUBLE)"

            params["stb_sy"] = int(start_year)
            params["stb_ey"] = int(end_year)
            where_clauses = [
                f"{yr_expr} BETWEEN %(stb_sy)s AND %(stb_ey)s",
                f"(UPPER(CAST({meas_col} AS STRING)) LIKE '%MID-MONTH%' "
                f"OR UPPER(CAST({meas_col} AS STRING)) LIKE '%MID MONTH%' "
                f"OR UPPER(TRIM(CAST({meas_col} AS STRING))) = 'EMPLOYEES MID-MONTH')",
                f"UPPER(TRIM(CAST({sub_col} AS STRING))) NOT LIKE 'TOTAL%'",
                f"UPPER(TRIM(CAST({sub_col} AS STRING))) NOT LIKE 'ALL %'",
            ]

            if stb_mark:
                params["stb_mark"] = stb_mark
                where_clauses.append(f"UPPER(TRIM(CAST({aar_col} AS STRING))) = %(stb_mark)s")

            if sub_measure_prefix:
                params["stb_sub"] = f"{sub_measure_prefix.strip().upper()}%"
                where_clauses.append(f"UPPER(TRIM(CAST({sub_col} AS STRING))) LIKE %(stb_sub)s")

            where_sql = " AND ".join(where_clauses)
            fq_stb = config.fq_table(config.stb_1246_table)

            q_by_carrier = f"""
            WITH monthly_carrier AS (
                SELECT
                    UPPER(TRIM(CAST({aar_col} AS STRING))) AS stb_aar_mark,
                    {yr_expr} AS year,
                    {mo_expr} AS month,
                    SUM({num_val_expr}) AS monthly_employees
                FROM {fq_stb} e
                WHERE {where_sql}
                GROUP BY 1, 2, 3
            )
            SELECT
                stb_aar_mark,
                COUNT(*) AS months_reported,
                ROUND(AVG(monthly_employees), 1) AS avg_monthly_employees,
                ROUND(MIN(monthly_employees), 0) AS min_monthly_employees,
                ROUND(MAX(monthly_employees), 0) AS max_monthly_employees
            FROM monthly_carrier
            GROUP BY stb_aar_mark
            ORDER BY avg_monthly_employees DESC
            """

            q_by_craft = f"""
            WITH monthly_craft AS (
                SELECT
                    TRIM(CAST({sub_col} AS STRING)) AS craft_category,
                    {yr_expr} AS year,
                    {mo_expr} AS month,
                    SUM({num_val_expr}) AS monthly_employees
                FROM {fq_stb} e
                WHERE {where_sql}
                GROUP BY 1, 2, 3
            )
            SELECT
                craft_category,
                ROUND(AVG(monthly_employees), 1) AS avg_monthly_employees
            FROM monthly_craft
            GROUP BY craft_category
            ORDER BY craft_category ASC
            """

            carrier_rows = execute_query(q_by_carrier, params)
            craft_rows = execute_query(q_by_craft, params)
            total_avg_employees = round(sum(float(r["avg_monthly_employees"] or 0) for r in carrier_rows), 1)

            fra_safety_rates: Dict[str, Any] = {
                "fra_parent_railroad_matched": fra_parent,
                "train_accidents_form54": None,
                "employee_on_duty_casualties_form55a": None,
                "train_accidents_per_1000_employees": None,
                "employee_casualties_per_1000_employees": None,
            }

            if not state and total_avg_employees > 0:
                p_fra: Dict[str, Any] = {}
                c54 = get_table_columns(config.form_54_table)
                inc_col = resolve_col(c54, "f", ("incident_key", "accident_number"))
                jcd_col = resolve_col(c54, "f", ("joint_cd",))
                t54 = build_time_predicate(
                    c54, "f", "st54", p_fra, ("year", "accident_year"), ("accident_month",), ("date",),
                    start_year, end_year, "calendar",
                )
                cls_filter = None if fra_parent else "1"
                f54_c = build_common_filters(
                    c54, "f", "sc54", p_fra, False, None, None, None, None, None,
                    None, fra_parent, cls_filter, None,
                )
                f54_x = build_form54_filters(c54, "f", "sx54", p_fra, include_crossings=False)
                q54 = f"""
                SELECT COUNT(DISTINCT CASE WHEN TRIM(CAST({jcd_col} AS STRING)) = '1' THEN {inc_col} END) AS cnt
                FROM {config.fq_table(config.form_54_table)} f
                WHERE {t54} {f54_c} {f54_x}
                """

                c55a = get_table_columns(config.form_55a_table)
                pt_col = resolve_col(c55a, "a", ("type_of_person", "type_of_person_code"))
                t55a = build_time_predicate(
                    c55a, "a", "st55a", p_fra, ("incident_year", "year"), ("incident_month",), ("date",),
                    start_year, end_year, "calendar",
                )
                f55a_c = build_common_filters(
                    c55a, "a", "sc55a", p_fra, False, None, None, None, None, None,
                    None, fra_parent, cls_filter, None,
                )
                q55a = f"""
                SELECT COUNT(*) AS cnt
                FROM {config.fq_table(config.form_55a_table)} a
                WHERE {t55a} {f55a_c}
                  AND (UPPER(CAST({pt_col} AS STRING)) LIKE '%EMPLOYEE ON DUTY%' OR TRIM(CAST({pt_col} AS STRING)) = 'A')
                """

                acc_cnt = int(execute_query(q54, p_fra)[0]["cnt"] or 0)
                emp_cas_cnt = int(execute_query(q55a, p_fra)[0]["cnt"] or 0)

                fra_safety_rates.update({
                    "train_accidents_form54": acc_cnt,
                    "employee_on_duty_casualties_form55a": emp_cas_cnt,
                    "train_accidents_per_1000_employees": round((acc_cnt / total_avg_employees) * 1000, 2),
                    "employee_casualties_per_1000_employees": round((emp_cas_cnt / total_avg_employees) * 1000, 2),
                })

            subsidiary_note = (
                "STB GTC (Grand Trunk Corp.) reflects CN's U.S. Class I subsidiary operations only; "
                "FRA parent_railroad_code='CN' aggregates CN's U.S. operating subsidiaries (GTW, IC, WC)."
                if stb_mark == "GTC"
                else "STB Part 1246 covers U.S. Class I railroad operations at the system level (no state breakdown)."
            )

            return {
                "dataset": f"STB 49 CFR Part 1246 Monthly Employment ({config.stb_1246_table})",
                "filters_applied": {
                    "start_year": start_year,
                    "end_year": end_year,
                    "requested_railroad_code": railroad_code,
                    "resolved_stb_aar_mark": stb_mark,
                    "resolved_fra_parent_code": fra_parent,
                    "sub_measure_prefix": sub_measure_prefix,
                    "state": state,
                },
                "total_avg_monthly_employees": total_avg_employees,
                "carrier_employment_summary": carrier_rows,
                "craft_breakdown_l100_to_l600": craft_rows,
                "cross_form_fra_safety_rates": fra_safety_rates,
                "regulatory_scope_note": subsidiary_note,
            }
        except DatabaseQueryError as err:
            return {"error": err.message, "hint": err.hint}
'''

TOOLS_INIT_CODE = '''from fastmcp import FastMCP

from . import (
    corridor_safety,
    cross_form_query,
    delta_audit,
    grade_crossings,
    joint_cause_audit,
    report_1_12,
    stb_employment,
    train_accidents,
)


def register_tools(mcp: FastMCP) -> None:
    report_1_12.register(mcp)
    train_accidents.register(mcp)
    joint_cause_audit.register(mcp)
    grade_crossings.register(mcp)
    corridor_safety.register(mcp)
    delta_audit.register(mcp)
    cross_form_query.register(mcp)
    stb_employment.register(mcp)


register_all_tools = register_tools
'''

APP_YAML_CONTENT = '''command:
  - "uvicorn"
  - "deploy.databricks.app:app"
  - "--host"
  - "0.0.0.0"
  - "--port"
  - "8000"

env:
  - name: PYTHONPATH
    value: "src"
  - name: DATABRICKS_WAREHOUSE_ID
    valueFrom: sql-warehouse
  - name: FRA_CATALOG
    value: "users"
  - name: FRA_SCHEMA
    value: "neeraj_koul"
'''

VERIFY_ENDPOINT_CONTENT = '''import asyncio
import concurrent.futures
import json
from pathlib import Path
import sys
import urllib.request

sys.dont_write_bytecode = True

REPO_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = str(REPO_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from databricks.sdk import WorkspaceClient
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport
from fra_safety_mcp.config import config

APP_BASE_URL = config.app_url.rstrip("/")
if not APP_BASE_URL:
    raise RuntimeError("MCP_APP_URL is not set. Add MCP_APP_URL=https://<your-app>.aws.databricksapps.com to .env")
MCP_ENDPOINT = f"{APP_BASE_URL}/mcp"


async def _check_live_app() -> None:
    w = WorkspaceClient()
    auth_headers = w.config.authenticate()

    req = urllib.request.Request(f"{APP_BASE_URL}/", headers=auth_headers)
    with urllib.request.urlopen(req, timeout=20) as resp:
        print("Root Health (/):", json.dumps(json.loads(resp.read().decode("utf-8"))))

    transport = StreamableHttpTransport(url=MCP_ENDPOINT, headers=auth_headers)
    async with Client(transport) as client:
        tools = [t.name for t in await client.list_tools()]
        resources = [str(r.uri) for r in await client.list_resources()]
        prompts = [p.name for p in await client.list_prompts()]
        print(f"Live /mcp Primitives -> Tools: {len(tools)}, Resources: {len(resources)}, Prompts: {len(prompts)}")

        res = await client.call_tool(
            "get_report_1_12_rail_safety_overview",
            {"start_year": 2023, "end_year": 2024, "parent_railroad_code": "CN"},
        )
        assert "error" not in res.data, f"Live app SQL error (check GRANTs): {res.data}"
        print("Live /mcp Report 1.12 Query: PASS")


def verify_live() -> None:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(asyncio.run, _check_live_app()).result()


if __name__ == "__main__":
    verify_live()
'''

README_CONTENT = '''# mcp-fra-safety

A Model Context Protocol (MCP) server built with **FastMCP** and **Databricks Unity Catalog** providing governed, regulatory-accurate analytical access to **Federal Railroad Administration (FRA) 49 CFR Part 225** safety datasets and **Surface Transportation Board (STB) 49 CFR Part 1246** Class I monthly employment data.

## Regulatory & Cross-Agency Rules Enforced

- **Multi-Train Collision Deduplication:** Counts distinct `incidentkey` records where `jointcd = '1'` for macro totals while summing `totaldamagecost` across all involved railroad filings.
- **Geographic Blank-Rate Rule (Report 1.12 & STB 1246):** Automatically returns `None` for Form 55 / STB 1246 system-level denominators and normalized rates whenever state, county, or city filters are active.
- **Calendar & Federal Fiscal Year Support:** Supports both Calendar Year (Jan-Dec) and Federal Fiscal Year (Oct 1 - Sep 30) across FRA forms.
- **Cross-Form & STB AAR Mark Linkage:** Walks linked incidents across Form 54 (`incidentkey`, `gxid`), Form 57 (`gradecrossingid`), Form 55a (casualties), and STB Part 1246 (`stb_1246_all_carriers`, crosswalking `CSXT` <-> `CSX`, `GTC` <-> `CN` U.S., `AMTK` <-> `ATK`).

## MCP Primitives (9 Tools, 6 Resources, 4 Prompts)

- `get_report_1_12_rail_safety_overview` — Replicates FRA Report 1.12 overview & normalized rates across Forms 54, 55a, 57, and 55.
- `get_train_accident_summary` — Replicates Form 54 Reports 2.03, 3.10 & 3.18 (Table 1 macro & Table 2 carrier overlap).
- `audit_joint_track_cause_discrepancies` — Self-joins Form 54 collisions to surface conflicting cause codes & speeds.
- `analyze_highway_rail_crossings` — Replicates Form 57 Reports 1.11, 2.08, 5.14, 5.15 & USDOT crossing hotspots.
- `analyze_metro_corridor_safety` — Synthesizes multi-portal rail & crossing risk across metropolitan corridors.
- `audit_delta_table_versions` — Audits Unity Catalog Delta `DESCRIBE HISTORY` governance & retroactive updates.
- `walk_across_fra_forms` — Walks linked incidents across Forms 54, 57, and 55a.
- `run_fra_sql_query` — Executes governed read-only cross-table SQL `SELECT`/`WITH` queries with truncation telemetry.
- `analyze_class1_employment_and_safety` — Analyzes STB Part 1246 Class I headcounts (`L100`-`L600`) and FRA safety rates.

## Quickstart

1. Copy `.env.example` to `.env` and populate your Databricks Unity Catalog and SQL Warehouse values:
   ```bash
   cp .env.example .env
